In [ ]:
import os, json
import pandas as pd
import numpy as np

from src.distribution_calibration import DistributionCalibration, DistributionCalibrationBatch

In [ ]:
data_dir = os.path.join("..", "..", "data")
opinionqa_dir = os.path.join(data_dir, "OpinionQA", "distribution_calibration")
dataset_name = "OpinionQA"

# Read and prepare data

In [ ]:
with open(os.path.join(opinionqa_dir, 'Qs_likert_scale_5_choices.json'), 'r') as f:
    Qs = json.load(f)

persona_answers = pd.read_csv(os.path.join(opinionqa_dir, 'persona_answers.csv'), index_col=0)

# Answer grid
possible_ratings = [1, 2, 3, 4, 5]

# Create dataframe with answer counts
dataset_counts = np.array([list(Qs[q]['choice_counts'].values()) for q in Qs.keys()])

# Convert to probabilities by dividing each row by its sum
P = dataset_counts / dataset_counts.sum(axis=1)[:, np.newaxis]

# Get digital twin ratings
Y_hat = persona_answers.values

# Load personas to examine top personas
with open(os.path.join(data_dir, 'personas.json'), 'r') as f:
    personas = json.load(f)

# get the list of pids in persona_answers
pids = list(persona_answers.columns)

# One run

In [ ]:
# Create optimizer
opt = DistributionCalibration(
    P, 
    Y_hat, 
    dataset_name,
    personas,
    pids,
    possible_ratings, 
    divergence='l1', 
    method='mirror_descent', 
    reg_w=1e-6, 
    reg_v=1e-6, 
    reg_mse=1e-7, 
    fit_persona_only=False, 
    fit_dummy_only=False, 
    weight_tol=None,
    max_iter=100000, 
    tol=1e-5, 
    learning_rate=1e-2, 
    train_test_ratio=0.8, 
    random_state=42,
    adaptive_lr=True, 
    max_grad_norm=10.0
)

# Run full workflow
results = opt.run_full_workflow(
    plot_convergence=True,
    plot_variance_ratio=True,
    plot_dist_difference=True,
    test_question_idx=0,
    get_top_weights=True,
    top_num=10,
    plot_divergence_comparison=True
)

# Multiple runs

In [ ]:
opt_batch = DistributionCalibrationBatch(
    P,
    Y_hat,
    dataset_name,
    personas,
    pids,
    possible_ratings,
    divergences=['tv', 'chi2', 'kl', 'hellinger', 'ks', 'l1', 'l2'],
    method='mirror_descent',
    reg_w=1e-6,
    reg_v=1e-6,
    reg_mse=1e-7,
    weight_tol=None,
    max_iter=100000,
    tol=1e-5,
    learning_rate=1e-2,
    train_test_ratio=0.8,
    random_state=42,
    adaptive_lr = True,
    max_grad_norm = 10.0,
)

results = opt_batch.run_all_experiments()